# DataForge AI — Automated Data Preparation Notebook
### End-to-End Machine Learning Data Preprocessing Pipeline

This notebook demonstrates the complete data preparation pipeline on the **UCI Adult Census Income Dataset** (32,561 records) adhering to scientific reproducibility standards.

**Pipeline Structure:**
1. Ingestion & Provenance Auditing
2. Exploratory Data Profiling & Sentinel Token Resolution
3. Missing Value Imputation
4. Duplicate Row Detection & Pruning
5. Outlier Detection (IQR & Z-Score) and Winsorization
6. Feature Engineering (Domain Features)
7. Numerical Scaling & Categorical Encoding
8. Final Validation & Clean Dataset Export

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, LabelEncoder

print("Libraries successfully imported.")

## 1. Ingestion & Raw Staging

In [ ]:
raw_path = '../data/raw/adult.csv'
df = pd.read_csv(raw_path)
print(f"Raw dataset staged: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head(5)

## 2. Missing Value Sentinel Replacement & Audit

In [ ]:
# In the UCI Adult dataset, missing entries are encoded as '?'
for col in df.select_dtypes(include=['object', 'string']).columns:
    df[col] = df[col].replace({'?': np.nan, '': np.nan})

missing_counts = df.isna().sum()
missing_summary = pd.DataFrame({
    'Missing Count': missing_counts,
    'Percentage (%)': (missing_counts / len(df)) * 100
}).query('`Missing Count` > 0')

missing_summary

## 3. Missing Value Imputation

In [ ]:
# Categorical mode imputation
for col in ['workclass', 'occupation', 'native_country']:
    mode_val = df[col].mode()[0]
    df[col] = df[col].fillna(mode_val)
    print(f"Imputed '{col}' with mode: {mode_val}")

print(f"Remaining null values across dataset: {df.isna().sum().sum()}")

## 4. Duplicate Record Detection & Pruning

In [ ]:
duplicates_found = df.duplicated().sum()
print(f"Detected {duplicates_found} duplicate records.")
df = df.drop_duplicates(keep='first')
print(f"Shape after deduplication: {df.shape}")

## 5. Outlier Detection & IQR Winsorization

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
for col in num_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    df[col] = df[col].clip(lower=lower, upper=upper)
    print(f"{col}: {outliers} outliers capped between [{lower:.1f}, {upper:.1f}]")

## 6. Feature Engineering (Domain Features)

In [ ]:
# 1. Age Groups
df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 25, 45, 65, 120],
    labels=['Youth', 'Early-Career', 'Mid-Career', 'Retirement']
).astype(str)

# 2. Work Schedule Tiers
df['work_schedule_type'] = pd.cut(
    df['hours_per_week'],
    bins=[0, 35, 40, 50, 168],
    labels=['Part-Time', 'Standard', 'Extended', 'Overtime']
).astype(str)

# 3. Capital Gain / Loss Binary Indicators
df['has_capital_gain'] = (df['capital_gain'] > 0).astype(int)
df['has_capital_loss'] = (df['capital_loss'] > 0).astype(int)

print(f"Shape after feature engineering: {df.shape}")
df[['age', 'age_group', 'hours_per_week', 'work_schedule_type', 'has_capital_gain']].head(5)

## 7. Numerical Scaling & Clean Export

In [ ]:
scaler = StandardScaler()
scale_cols = ['age', 'fnlwgt', 'education_num', 'hours_per_week']
df[scale_cols] = np.round(scaler.fit_transform(df[scale_cols]), 4)

output_file = '../data/processed/adult_cleaned.csv'
df.to_csv(output_file, index=False)
print(f"Sanitized dataset successfully saved to: {output_file}")
print(f"Final clean shape: {df.shape[0]:,} rows × {df.shape[1]} columns")